<span style="color: #6a737d; font-family: monospace;">
Created on Tue Feb 11 2025 16:59:06<br>
Author: Mukai (Tom Notch) Yu<br>
Email: mukaiy@andrew.cmu.edu<br>
Affiliation: Carnegie Mellon University, Robotics Institute<br>
<br>
Copyright Ⓒ 2025 Mukai (Tom Notch) Yu<br>
</span>

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "4"

# %cd $USF_PROJECT_DIRECTORY doesn't work here because it's set by os.environ, not before the notebook starts
%cd ../..
%load_ext autoreload
%autoreload 2

from copy import deepcopy

from hydra import compose, initialize
from hydra.utils import instantiate
from pytorch_lightning import Trainer, callbacks, loggers

from usf.network.model.object_detection import ObjectDetectionLightningModel
from usf.utils.files import read_file
from usf.utils.torch_numpy import fixed_seed, string_to_seed

/home/mukaiy/code/spherical-model/USF


In [2]:
batch_size = 2

## Read Config

In [3]:
with initialize(
    version_base=None,
    config_path="../../config",
):  # hydra doesn't respect the current working directory
    config = compose(
        config_name="default.yaml",
        overrides=[
            "task=object_detection",
            "task.data_module.num_workers=1",
            f"task.data_module.batch_size={batch_size}",
            "~task.data_module.train_augmentation",  # disable training augmentation
            "task.trainer.log_every_n_steps=1",
            "~task.trainer.strategy",  # no strategy
            "~task.trainer.logger",  # disable logging
            "~task.trainer.profiler",  # disable profiler
            "+task.trainer.enable_progress_bar=True",  # print progress in notebook
            "+task.trainer.enable_checkpointing=False",  # disable checkpointing,
        ],
    )

## Single Batch Overfit Test

In [4]:
datamodule = instantiate(config.task.data_module)
datamodule.setup()  # must setup() because that instantiates self.train_dataset and self.val_dataset

# ensure same sample
with fixed_seed(string_to_seed("Object Detection")):
    panoramic_single_batch_datamodule = batch_size @ datamodule

panoramic_single_batch_datamodule.val_dataloader = (
    lambda: []
)  # override val_dataloader function to provide empty validation dataloader to skip validation
panoramic_single_batch_datamodule.predict_dataloader = (
    lambda: panoramic_single_batch_datamodule.train_dataloader()
)

In [5]:
pinhole_single_batch_datamodule = deepcopy(panoramic_single_batch_datamodule)
pinhole_single_batch_datamodule.predict_dataloader = (
    lambda: pinhole_single_batch_datamodule.train_dataloader()
)
pinhole_single_batch_datamodule.train_dataset.set_output_vector(
    read_file("config/lens_normal_map/90_90_640_640.npy")
)

In [6]:
fisheye_single_batch_datamodule = deepcopy(panoramic_single_batch_datamodule)
fisheye_single_batch_datamodule.predict_dataloader = (
    lambda: fisheye_single_batch_datamodule.train_dataloader()
)
fisheye_single_batch_datamodule.train_dataset.set_output_vector(
    read_file("config/lens_normal_map/180_180_640_640.npy")
).set_output_vector_mask(read_file("config/lens_normal_map/180_180_640_640_mask.npy"))

In [7]:
with initialize(version_base=None, config_path="../../config"):
    planar_config = compose(
        config_name="default.yaml",
        overrides=[
            "task=object_detection",
            "task/object_detection@task.architecture=planar",
        ],
    )
    planar_model: ObjectDetectionLightningModel = instantiate(planar_config.task.model)

In [8]:
with initialize(version_base=None, config_path="../../config"):
    spherical_config = compose(
        config_name="default.yaml",
        overrides=[
            "task=object_detection",
            "task/object_detection@task.architecture=spherical",
        ],
    )
    spherical_model: ObjectDetectionLightningModel = instantiate(
        spherical_config.task.model
    )

### Train Planar Model

In [ ]:
trainer_kwargs = {
    "callbacks": [
        callbacks.LearningRateMonitor(logging_interval="epoch"),
        # FreezeBatchNorm(),
    ],
    "logger": loggers.TensorBoardLogger(
        save_dir="tensorboard_logs",
        name=f"{planar_model.model.backbone.__class__.__name__} Object Detection",
    ),
    "max_epochs": 200,
}
trainer: Trainer = instantiate(config.task.trainer, **trainer_kwargs)

In [ ]:
trainer.fit(planar_model, panoramic_single_batch_datamodule)

### Train Spherical Model

In [9]:
# re-instantiate the trainer to reset the state
trainer_kwargs = {
    "callbacks": [
        callbacks.LearningRateMonitor(logging_interval="epoch"),
        # FreezeBatchNorm(),
    ],
    "logger": loggers.TensorBoardLogger(
        save_dir="tensorboard_logs",
        name=f"{spherical_model.model.backbone.__class__.__name__} Object Detection",
    ),
    "max_epochs": 200,
}
trainer: Trainer = instantiate(config.task.trainer, **trainer_kwargs)

/home/mukaiy/miniforge3/envs/usf/lib/python3.12/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/mukaiy/miniforge3/envs/usf/lib/python3.12/site ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [10]:
trainer.fit(spherical_model, panoramic_single_batch_datamodule)

You are using a CUDA device ('NVIDIA A100-SXM4-80GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [4]
Loading `train_dataloader` to estimate number of stepping batches.
/home/mukaiy/miniforge3/envs/usf/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/mukaiy/miniforge3/envs/usf/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=95` in the `DataLoader` to impr

┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ SphericalObjectDetection │ 24.4 M │ train │     0 │
└───┴───────┴──────────────────────────┴────────┴───────┴───────┘

Trainable params: 24.4 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.4 M                                                                                               
Total estimated model params size (MB): 97                                                                         
Modules in train mode: 1521                                                                                        
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/mukaiy/miniforge3/envs/usf/lib/python3.12/site-packages/pytorch_lightning/utilities/data.py:106: Total length of `list` across ranks is zero. Please make sure this was your intention.


/home/mukaiy/code/spherical-model/USF/usf/network/layer/spherical/_old/circle_cnn.py:285: UserWarning: Sparse 
invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse 
tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly 
opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at 
/pytorch/aten/src/ATen/Context.cpp:760.)
  collection_matrix = torch.sparse_coo_tensor(

AssertionError: inconsistent geometry between predict_maps and gt_maps

## Visualize Ground Truth and Prediction

In [ ]:
inference_model = spherical_model

### Panoramic

In [ ]:
predictions: list[dict] = trainer.predict(
    inference_model,
    panoramic_single_batch_datamodule,
    return_predictions=True,
)

In [ ]:
prediction_batch = predictions[0]

In [ ]:
gt_figures, gt_spherical_vis = (
    panoramic_single_batch_datamodule.train_dataset.visualize_batch(
        batch=prediction_batch
    )
)

In [ ]:
prediction_batch["predicts"]["converted_rbfovs"] = (
    panoramic_single_batch_datamodule.train_dataset.extract_raw_rbfovs(
        prediction_batch["predicts"]["maps"],
        probability_threshold=0.3,
        pool_radius=0.06705,
    )
)

In [ ]:
prediction_batch["predicts"]["converted_rbfovs"] = (
    panoramic_single_batch_datamodule.train_dataset.non_maximum_suppression(
        prediction_batch["predicts"]["converted_rbfovs"],
        max_rbfov_per_category=20,
        sigma=5.0,
        score_threshold=0.3,
        reduction="prod",
        compensate=False,
        hard_iou_threshold=0.2,
        epsilon_tie_break=1e-6,
        vector_average_area=2.7e-5,
    )
)

In [ ]:
predict_figures, predict_spherical_vis = (
    panoramic_single_batch_datamodule.train_dataset.visualize_batch(
        batch={
            "inputs": prediction_batch["inputs"],
            "labels": prediction_batch["predicts"],
            "meta": prediction_batch["meta"],
        },
    )
)

### Pinhole

In [ ]:
with fixed_seed(string_to_seed("Object Detection")):
    pinhole_predictions: list[dict] = trainer.predict(
        inference_model,
        pinhole_single_batch_datamodule,
        return_predictions=True,
    )

In [ ]:
pinhole_prediction_batch = pinhole_predictions[0]

In [ ]:
gt_figures, gt_spherical_vis = (
    pinhole_single_batch_datamodule.train_dataset.visualize_batch(
        batch=pinhole_prediction_batch
    )
)

In [ ]:
pinhole_prediction_batch["predicts"]["converted_rbfovs"] = (
    pinhole_single_batch_datamodule.train_dataset.extract_raw_rbfovs(
        pinhole_prediction_batch["predicts"]["maps"],
        probability_threshold=0.25,
        pool_radius=0.06705,
    )
)

In [ ]:
pinhole_prediction_batch["predicts"]["converted_rbfovs"] = (
    pinhole_single_batch_datamodule.train_dataset.non_maximum_suppression(
        pinhole_prediction_batch["predicts"]["converted_rbfovs"],
        max_rbfov_per_category=20,
        sigma=5.0,
        score_threshold=0.3,
        reduction="prod",
        compensate=False,
        hard_iou_threshold=0.2,
        epsilon_tie_break=1e-6,
        vector_average_area=2.7e-5,
    )
)

In [ ]:
predict_figures, predict_spherical_vis = (
    pinhole_single_batch_datamodule.train_dataset.visualize_batch(
        batch={
            "inputs": pinhole_prediction_batch["inputs"],
            "labels": pinhole_prediction_batch["predicts"],
            "meta": pinhole_prediction_batch["meta"],
        },
    )
)

### Fisheye

In [ ]:
with fixed_seed(string_to_seed("Object Detection")):
    fisheye_predictions: list[dict] = trainer.predict(
        inference_model,
        fisheye_single_batch_datamodule,
        return_predictions=True,
    )

In [ ]:
fisheye_prediction_batch = fisheye_predictions[0]

In [ ]:
gt_figures, gt_spherical_vis = (
    fisheye_single_batch_datamodule.train_dataset.visualize_batch(
        batch=fisheye_prediction_batch
    )
)

In [ ]:
fisheye_prediction_batch["predicts"]["converted_rbfovs"] = (
    fisheye_single_batch_datamodule.train_dataset.extract_raw_rbfovs(
        fisheye_prediction_batch["predicts"]["maps"],
        probability_threshold=0.3,
        pool_radius=0.06705,
    )
)

In [ ]:
fisheye_prediction_batch["predicts"]["converted_rbfovs"] = (
    fisheye_single_batch_datamodule.train_dataset.non_maximum_suppression(
        fisheye_prediction_batch["predicts"]["converted_rbfovs"],
        max_rbfov_per_category=20,
        sigma=5.0,
        score_threshold=0.3,
        reduction="prod",
        compensate=False,
        hard_iou_threshold=0.2,
        epsilon_tie_break=1e-6,
        vector_average_area=2.7e-5,
    )
)

In [ ]:
predict_figures, predict_spherical_vis = (
    fisheye_single_batch_datamodule.train_dataset.visualize_batch(
        batch={
            "inputs": fisheye_prediction_batch["inputs"],
            "labels": fisheye_prediction_batch["predicts"],
            "meta": fisheye_prediction_batch["meta"],
        },
    )
)